In [16]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# 參數設定

In [17]:
# ===== 路徑 =====
PILOT_CSV  = "/home/yingtzudai/Multimodal_SPoD_Video/data/split/pilot_posts.csv"
TEXT_DIR   = "/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer/"            # 內含 {username}.csv
OUTPUT_DIR = "/home/yingtzudai/Multimodal_SPoD_Video/features/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== 欄位名稱（請依 pilot_posts.csv 實際欄名調整）=====
USER_COL  = "username"   # pilot 中網紅帳號欄位
ID_COL    = "media_id"
TEXT_COL  = "text"
LABEL     = "label"
# ===== 模型參數 =====
# 論文用 BERT；IG 貼文常中英混雜，預設用 multilingual 版本
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN    = 512          # BERT 上限（含 [CLS]、[SEP]）
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


# 讀取資料

In [18]:
# media_id 一律以字串讀入：IG media_id 是 18～19 位數，用 float 讀會失去精度
pilot = pd.read_csv(PILOT_CSV, dtype={ID_COL: str})
print(pilot.shape)
print(pilot.columns.tolist())

def load_texts(pilot_df):
    """每位網紅的 csv 只讀一次，再用 media_id 對回 text"""
    id2text = {}
    for user, sub in tqdm(pilot_df.groupby(USER_COL), desc="Loading text csv"):
        path = os.path.join(TEXT_DIR, f"{user}-FullVideoInfo.csv")
        if not os.path.exists(path):
            print(f"[WARN] missing text file: {path}")
            continue
        tdf = pd.read_csv(path, dtype={ID_COL: str})
        tdf = tdf[tdf[ID_COL].isin(set(sub[ID_COL]))]
        id2text.update(dict(zip(tdf[ID_COL], tdf[TEXT_COL])))
    return id2text

id2text = load_texts(pilot)
pilot[TEXT_COL] = pilot[ID_COL].map(id2text).fillna("").astype(str)

n_missing = (pilot[TEXT_COL].str.strip() == "").sum()
print(f"matched: {len(id2text)} / {len(pilot)}, empty text: {n_missing}")
pilot[[ID_COL, USER_COL, TEXT_COL]].head()

(299, 7)
['media_id', 'id', 'username', 'person_id', 'stem', 'label', 'split']


Loading text csv:   0%|          | 0/91 [00:00<?, ?it/s]

matched: 299 / 299, empty text: 1


,media_id,username,text
0,1481312756980880,uccu0323,終於敢開口唱歌了🥹🎵...\n前提是\n要有 @ah_miu0810 在我旁邊♥️
1,786977131086981,elephant_gogo,2025 第二次進手術房，真的沒事不要再進了\n\n很多人都問我為什麼要再次取卵？\n我這邊...
2,1263468302496427,howhowhasfriends,拿這顆精靈球去收服朋友吧！\n\n哪有這麼可愛的佛跳牆啦～\n做工超精緻 趕快收藏！\n\...
3,1430413305494426,donehannah,小時候的味道～🐷🤎\n以前董嬤來我們家，都會炸一鍋豬油\n讓我爸做飯給我跟妹妹吃的時候使用\...
4,698570256678772,cocowine0205,🩵不知道就太落伍啦！討論度爆高的新品！\n💰青花椒餅乾289元（20入）\n沒想到喜年來會出...


# Text Encoder

## Fundalmental Setting of Text Encoder

In [19]:
class TextEncoder(nn.Module):
    """
    SPoD §3.2.2
    c  = [c1, ..., cj]
    c' = [[CLS], c'1, ..., c'j']           (tokenizer 自動加 [CLS])
    D^o = F_T(F_T^{o-1}) = [d^o_[CLS], d^o_1, ..., d^o_j']
    X^T = d^o_[CLS]                         (只取最後一層的 [CLS])
    """
    def __init__(self, model_name=MODEL_NAME, freeze=True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_dim = self.bert.config.hidden_size   # 768
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        D_o = out.last_hidden_state        # (B, j'+1, H)：第 o 層輸出
        X_T = D_o[:, 0, :]                 # (B, H)：d^o_[CLS]
        return X_T

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = TextEncoder(MODEL_NAME, freeze=True).to(DEVICE).eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 批次產生Text Embedding

In [20]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE):
    feats = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        X_T = encoder(**enc)
        feats.append(X_T.cpu())
    return torch.cat(feats, dim=0)

texts = pilot[TEXT_COL].tolist()
X_T = encode_texts(texts)
print(X_T.shape)   # (N, 768)

Encoding:   0%|          | 0/10 [00:00<?, ?it/s]

torch.Size([299, 768])


In [21]:
save_path = os.path.join(f"{OUTPUT_DIR}/text/", "pilot_text_cls.pt")
torch.save({
    "media_id": pilot[ID_COL].tolist(),
    "stem": pilot["stem"].tolist() if "stem" in pilot.columns else None,
    "X_T": X_T,                    # (N, 768) float tensor
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
}, save_path)
print("saved →", save_path)

saved → /home/yingtzudai/Multimodal_SPoD_Video/features//text/pilot_text_cls.pt


In [22]:
# 1. 有沒有 NaN
print("NaN:", torch.isnan(X_T).any().item())

# 2. 隨機兩篇貼文的 cosine similarity，看分布是否合理
import torch.nn.functional as F
idx = torch.randperm(len(X_T))[:200]
sim = F.cosine_similarity(X_T[idx].unsqueeze(1), X_T[idx].unsqueeze(0), dim=-1)
print("mean pairwise cos sim:", sim[~torch.eye(len(idx), dtype=bool)].mean().item())

NaN: False
mean pairwise cos sim: 0.8536259531974792
